# MNIST Digits Recognition - MLOps

## Kubeflow pipeline

In [1]:
%pip install kfp-kubernetes==1.4.0

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install keras

In [3]:
import kfp
import kfp.kubernetes
from kfp.dsl import pipeline, component

In [4]:
# Pipeline parameters
PARAMS = {
    "http_proxy": "",  # "http://<proxy_host>:<port>"
    "no_proxy": "",  # "localhost,.cluster.local,.svc,.default.svc, <host>:<port>, x.x.x.x/y,..."
    "user_shared_dir": "/mnt/user/",  # volume user-pvc, also mounted by this jupyter notebook
    "training_data_subdir": "training/",
    "model_subdir": "model",
    "model_trained_subdir": "model_trained",
    "export_model_dir": "/mnt/export",
    "num_epochs": 1,
    "lr": 0.1
}

In [5]:
# # Pre-run step in notebook
# from tensorflow import keras
# import numpy as np
# import os

# final_training_data_dir = PARAMS["user_shared_dir"] + PARAMS["training_data_subdir"]

# x_train_artifact = final_training_data_dir + "xtrain.npy"
# x_test_artifact = final_training_data_dir + "x_test.npy"
# y_train_artifact = final_training_data_dir + "ytrain.npy"
# y_test_artifact = final_training_data_dir + "y_test.npy"

# if not os.path.exists(final_training_data_dir):
#     os.makedirs(final_training_data_dir, 0o777)

# (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# np.save(x_train_artifact, x_train)
# np.save(y_train_artifact, y_train)
# np.save(x_test_artifact, x_test)
# np.save(y_test_artifact, y_test)

In [6]:
#BASE_IMG = "lr1-bd-harbor-registry.mip.storage.hpecorp.net/develop/gcr.io/mapr-252711/kubeflow/notebooks/jupyter-tensorflow-full:ezua-1.6.0-3733bdcf7"
#BASE_IMG = "172.28.1.47/ezmeral-common/gcr.io/mapr-252711/kubeflow/notebooks/jupyter-tensorflow-full:ezua-1.6.0-3733bdcf7"
BASE_IMG = "10.200.140.54/ezmeral-common/hpe-kubeflow/notebooks/jupyter-tensorflow-full:aie-1.11.0-9ba2ac0d"

In [7]:
@component(base_image=BASE_IMG, install_kfp_package=False)
def pre_processing(params: dict) -> str:
    # imports
    import numpy as np

    final_training_data_dir = params["user_shared_dir"] + params["training_data_subdir"]

    x_train_artifact = final_training_data_dir + "train-images-pickle.npy"
    x_test_artifact = final_training_data_dir + "test-images-pickle.npy"
    y_train_artifact = final_training_data_dir + "train-labels-pickle.npy"
    y_test_artifact = final_training_data_dir + "test-labels-pickle.npy"

    x_train_preproc = final_training_data_dir + "xtrain-preproc.npy"
    x_test_preproc = final_training_data_dir + "xtest-preproc.npy"

    # load data artifact store
    x_train = np.load(x_train_artifact) 
    x_test = np.load(x_test_artifact)

    # reshaping the data
    # reshaping pixels in a 28x28px image with greyscale, canal = 1. This is needed for the Keras API
    x_train = x_train.reshape(-1,28,28,1)
    x_test = x_test.reshape(-1,28,28,1)
    # normalizing the data
    # each pixel has a value between 0-255. Here we divide by 255, to get values from 0-1
    x_train = x_train / 255
    x_test = x_test / 255

    #logging metrics using Kubeflow Artifacts
    print("Len x_train", x_train.shape[0])
    print("Len y_train", x_test.shape[0])

    # save feuture in artifact store
    np.save(x_train_preproc, x_train)

    np.save(x_test_preproc, x_test)

    return "Step pre_processing is done"

In [8]:
@component(base_image=BASE_IMG, install_kfp_package=False)
def model_build(params: dict, prev_step: str) -> str:
    # imports
    import os
    import sys
    import subprocess
    lib_dir = "/tmp/custom_libs"
    os.makedirs(lib_dir, exist_ok=True)
    subprocess.check_call([
    "pip", "install",
    "--no-index",
    "--find-links", "/mnt/user/PCAI-Pipelines-main/02-Models-Operations/wheels",
    "--target", "/tmp/packages",
    "tensorflow==2.15.0",
    "kfp==2.0.0"
])
    import sys
    sys.path.insert(0, "/tmp/packages")
    from tensorflow import keras

    final_model_data_dir = params["user_shared_dir"] + params["model_subdir"]

    if not os.path.exists(final_model_data_dir):
        os.makedirs(final_model_data_dir, 0o777)

    model = keras.models.Sequential()
    model.add(keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28,28,1)))
    model.add(keras.layers.MaxPool2D(2, 2))

    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(64, activation='relu'))
    model.add(keras.layers.Dense(32, activation='relu'))

    model.add(keras.layers.Dense(10, activation='softmax'))

    #saving model
    model.save(final_model_data_dir)

    return "Step model_build is done"

In [9]:
@component(base_image=BASE_IMG, install_kfp_package=False)
def training(params: dict, prev_step: str) -> str:
    # imports
    import os
    import sys
    import subprocess
    lib_dir = "/tmp/custom_libs"
    os.makedirs(lib_dir, exist_ok=True)
    subprocess.check_call([
    "pip", "install",
    "--no-index",
    "--find-links", "/mnt/user/PCAI-Pipelines-main/02-Models-Operations/wheels",
    "--target", "/tmp/packages",
    "tensorflow==2.15.0",
    "kfp==2.0.0"
])
    import sys
    sys.path.insert(0, "/tmp/packages")
    from tensorflow import keras
    import os
    import numpy as np
    import tensorflow as tf

    model_trained_dir = params["user_shared_dir"] + params["model_trained_subdir"]

    if not os.path.exists(model_trained_dir):
        os.makedirs(model_trained_dir, 0o777)

    export_model_dir = params["export_model_dir"]

    #load dataset
    final_training_data_dir = params["user_shared_dir"] + params["training_data_subdir"]
    x_train_artifact = final_training_data_dir + "train-images-pickle.npy"
    x_test_artifact = final_training_data_dir + "test-images-pickle.npy"
    y_train_artifact = final_training_data_dir + "train-labels-pickle.npy"
    y_test_artifact = final_training_data_dir + "test-labels-pickle.npy"

    x_train_preproc = final_training_data_dir + "xtrain-preproc.npy"
    x_test_preproc = final_training_data_dir + "xtest-preproc.npy"

    x_train = np.load(x_train_preproc)
    x_test = np.load(x_test_preproc)
    y_train = np.load(y_train_artifact)
    y_test = np.load(y_test_artifact)

    #load model structure
    final_model_data_dir = params["user_shared_dir"] + params["model_subdir"]

    model = keras.models.load_model(final_model_data_dir)

    #compile the model - we want to have a binary outcome
    model.compile(tf.keras.optimizers.SGD(learning_rate=params["lr"]),
                  loss="sparse_categorical_crossentropy",
                  metrics=['accuracy'])

    #fit the model and return the history while training
    history = model.fit(
          x = x_train,
          y = y_train,
          epochs = params["num_epochs"],
          batch_size = 20,
    )

    # Test the model against the test dataset
    # Returns the loss value & metrics values for the model in test mode.
    model_loss, model_accuracy = model.evaluate(x=x_test,y=y_test)

    print(f"model_loss={model_loss} model_accuracy={model_accuracy} ")

    #build a confusione matrix
    y_predict = model.predict(x=x_test)
    y_predict = np.argmax(y_predict, axis=1)
    print(f"y_predict={y_predict}")

    #adding /1/ subfolder for TFServing
    model_trained_dir = model_trained_dir + '/1/'
    print("model_trained_uri: " + model_trained_dir)

    export_model_dir = export_model_dir + '/1/'
    print("model_export_uri: " + model_trained_dir)

    # save model in local storage
    keras.models.save_model(model, model_trained_dir, save_format='tf')

    # save model in model export volume
    keras.models.save_model(model, export_model_dir, save_format='tf')

    return "Step training is done"

In [10]:
@component(base_image=BASE_IMG, install_kfp_package=False)
def validation(params: dict, prev_step: str) -> str:
    # imports
    import os
    import sys
    import subprocess
    lib_dir = "/tmp/custom_libs"
    os.makedirs(lib_dir, exist_ok=True)
    subprocess.check_call([
    "pip", "install",
    "--no-index",
    "--find-links", "/mnt/user/PCAI-Pipelines-main/02-Models-Operations/wheels",
    "--target", "/tmp/packages",
    "tensorflow==2.15.0",
    "kfp==2.0.0"
    ])
    sys.path.insert(0, "/tmp/packages")
    from tensorflow import keras
    import numpy as np

    model_trained_dir = params["user_shared_dir"] + params["model_trained_subdir"]
    final_training_data_dir = params["user_shared_dir"] + params["training_data_subdir"]
    x_test_artifact = final_training_data_dir + "test-images-pickle.npy"
    y_test_artifact = final_training_data_dir + "test-labels-pickle.npy"
    
    x_test = np.load(x_test_artifact)
    y_test = np.load(y_test_artifact)

    #Load model from the volume
    model = keras.models.load_model(model_trained_dir + '/1/')

    test_accu = model.evaluate(x_test, y_test)
    print('The testing accuracy is :',test_accu[1]*100, '%')

    return "Step validation is done"

In [11]:
@component(base_image=BASE_IMG, install_kfp_package=False)
def kserve_serving(prev_step: str, namespace_cur: str) -> str:
    # imports
    from kubernetes import client, config, utils
    import subprocess
    import sys
    import os

    print("Does /mnt/user exist?", os.path.exists("/mnt/user"))
    print("Does wheels dir exist?", os.path.exists("/mnt/user/PCAI-Pipelines-main/02-Models-Operations/wheels"))

    if os.path.exists("/mnt/user/PCAI-Pipelines-main/02-Models-Operations/"):
        print("Contents:", os.listdir("/mnt/user/PCAI-Pipelines-main/02-Models-Operations/"))
    
    lib_dir = "/tmp/custom_libs"
    os.makedirs(lib_dir, exist_ok=True)
    subprocess.check_call([
    "pip", "install",
    "--no-index",
    "--find-links", "/mnt/user/PCAI-Pipelines-main/02-Models-Operations/wheels",
    "--target", "/tmp/packages",
    "tensorflow==2.15.0",
    "kfp==2.0.0"
    ])
    import sys
    sys.path.insert(0, "/tmp/packages")
    
    model_dir = "/mnt/export/"
    model_volume = "model-exported-mlops-pipeline"  
    model_name = "digitrec-serve"

    api_version = 'serving.kserve.io/v1beta1'

    inference_service_spec = {
        "apiVersion": api_version,
        "kind": "InferenceService",
        "metadata": {
            "name": model_name,
            "annotations": {
                "sidecar.istio.io/inject": "false"
            }
        },
        "spec": {
            "predictor": {
                "tensorflow": {
                    "storageUri": f"pvc://{model_volume}/",
                    "resources": {
                        "limits": {
                            "cpu": '2',
                            "memory": '1Gi'
                        },
                        "requests": {
                            "cpu": '100m',
                            "memory": '50Mi'
                        }
                    }
                }
            }
        }
    }

    config.load_incluster_config()
    api = client.CustomObjectsApi()
    api.create_namespaced_custom_object(
        group="serving.kserve.io",
        version="v1beta1",
        namespace=namespace_cur,
        plural="inferenceservices",
        body=inference_service_spec,
    )
    print("Resource created")

    return "Step kserve_serving is done"

In [12]:
import os

current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
current_sc

'gl4f-filesystem'

In [13]:
namespace_cur = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
namespace_cur

'project-user-pransshu-g-ext'

In [14]:
@pipeline(name='mlops-pipeline')
def mlops_pipeline():   
    pre_processing_task = pre_processing(params=PARAMS)

    #CT, 21/12/24: added those 4 limits to solve the OOMKilled issue
    pre_processing_task.set_memory_request("2Gi")
    pre_processing_task.set_memory_limit("4Gi")
    pre_processing_task.set_cpu_request("2")
    pre_processing_task.set_cpu_limit("4")

    
    kfp.kubernetes.mount_pvc(pre_processing_task, "user-pvc", PARAMS["user_shared_dir"])

    model_build_task = model_build(params=PARAMS, prev_step=pre_processing_task.output)
    kfp.kubernetes.mount_pvc(model_build_task, "user-pvc", PARAMS["user_shared_dir"])

    pvc = kfp.kubernetes.CreatePVC(
        pvc_name='model-exported-mlops-pipeline',
        access_modes=['ReadWriteMany'],
        size='5Gi',
        storage_class_name=current_sc,
    )
    #pvc.set_caching_options(False)

    training_task = training(params=PARAMS, prev_step=model_build_task.output)
    training_task.set_memory_request("4Gi")
    training_task.set_memory_limit("8Gi")
    training_task.set_cpu_request("4")
    training_task.set_cpu_limit("6")
    kfp.kubernetes.mount_pvc(training_task, "user-pvc", PARAMS["user_shared_dir"])
    kfp.kubernetes.mount_pvc(training_task, pvc.outputs['name'], PARAMS["export_model_dir"])

    validation_task = validation(params=PARAMS, prev_step=training_task.output)
    validation_task.set_memory_request("4Gi")
    validation_task.set_memory_limit("6Gi")
    validation_task.set_cpu_request("4")
    validation_task.set_cpu_limit("6")
    kfp.kubernetes.mount_pvc(validation_task, "user-pvc", PARAMS["user_shared_dir"])

    kserve_serving_task = kserve_serving(prev_step=validation_task.output, namespace_cur=namespace_cur)
    kserve_serving_task.set_caching_options(enable_caching=False)
    kfp.kubernetes.mount_pvc(kserve_serving_task, pvc.outputs['name'], PARAMS["export_model_dir"])
    kfp.kubernetes.mount_pvc(kserve_serving_task, "user-pvc", PARAMS["user_shared_dir"])

In [15]:
client = kfp.Client()
client.create_run_from_pipeline_func(mlops_pipeline)

/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


RunPipelineResult(run_id=dfb83086-07cb-4778-a35d-8399f0542774)

### Next step: Inference test

Go the the following step: [Inference notebook](03-Inference.ipynb)